In [1]:
import torch
import numpy as np
import sympy as sp
import os
import requests
import symbolicregression
import symbolicregression.model
from IPython.display import display


In [2]:
model_path = "model.pt"
if not os.path.isfile(model_path):
    url = "https://dl.fbaipublicfiles.com/symbolicregression/model1.pt"
    r = requests.get(url, allow_redirects=True)
    r.raise_for_status()
    open(model_path, "wb").write(r.content)
if not torch.cuda.is_available():
    model = torch.load(model_path, map_location=torch.device("cpu"), weights_only=False)
else:
    model = torch.load(model_path, weights_only=False)
    model = model.cuda()
print(model.device)
print("Model successfully loaded!")


cuda:0
Model successfully loaded!


In [3]:
est = symbolicregression.model.SymbolicTransformerRegressor(
                        model=model,
                        max_input_points=200,
                        n_trees_to_refine=100,
                        rescale=True
                        )

In [4]:
##Example of data

x = np.random.randn(100, 2)
y = np.cos(2*np.pi*x[:,0])+x[:,1]**2


In [5]:
est.fit(x,y)
replace_ops = {"add": "+", "mul": "*", "sub": "-", "pow": "**", "inv": "1/"}
model_str = est.retrieve_tree(with_infos=True)["relabed_predicted_tree"].infix()
for op,replace_op in replace_ops.items():
    model_str = model_str.replace(op,replace_op)
display(sp.parse_expr(model_str))

/home/xyh/Symbolic_Regression/E2E/E2E/symbolicregression/model/utils_wrapper.py:188: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.grad` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.func.grad` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  grad_obj = grad(objective_torch)(coeffs)


(0.12002718574873454*x_1 + 0.011819294811358543)*(-0.7419023398160064*x_0 + (8.14 + 0.751/(0.44343376957171374*x_1 + 10.643665728053074))*(1.0681308168806367*x_1 + 0.035010780029599025) - 0.8683623925681302) - 0.9650000000000001*sin(6.217647450162553*x_0 - 1.379423130465954) + 0.00681